Test autoregressive generation

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("chandar-lab/NovoMolGen_157M_SMILES_AtomWise", revision='hf-checkpoint', device_map='auto')
tokenizer = AutoTokenizer.from_pretrained("chandar-lab/NovoMolGen_157M_SMILES_AtomWise", revision='hf-checkpoint')

input_ids = torch.tensor([[tokenizer.bos_token_id]]).expand(4, -1).contiguous().to(model.device)
outs = model.generate(input_ids=input_ids, temperature=1.0, max_length=64, do_sample=True, pad_token_id=tokenizer.eos_token_id)

molecules = [t.replace(" ", "") for t in tokenizer.batch_decode(outs, skip_special_tokens=True)]
print(molecules)

test

In [1]:
import torch
import torch.nn as nn


class SpecToSmiles(nn.Module):
    """DreaMS encoder -> MLP projection -> prefix vectors -> decoder-only SMILES LM."""

    def __init__(self, encoder, decoder, d_enc=1024,
                 freeze_encoder=True, freeze_decoder=True, proj_dropout=0.1):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        d_dec = decoder.get_input_embeddings().weight.shape[1]
        self.proj = nn.Sequential(
            nn.Linear(d_enc, d_dec),
            nn.GELU(),
            nn.Dropout(proj_dropout),
            nn.Linear(d_dec, d_dec),
            nn.LayerNorm(d_dec),   # keeps prefix scale close to token-embedding scale
        )
        self.set_frozen(freeze_encoder, freeze_decoder)

    def set_frozen(self, freeze_encoder, freeze_decoder):
        self.freeze_encoder, self.freeze_decoder = freeze_encoder, freeze_decoder
        for p in self.encoder.parameters():
            p.requires_grad = not freeze_encoder
        for p in self.decoder.parameters():
            p.requires_grad = not freeze_decoder

    def train(self, mode=True):
        super().train(mode)
        if self.freeze_encoder:
            self.encoder.eval()      # no dropout in frozen parts
        if self.freeze_decoder:
            self.decoder.eval()
        return self

    def encode(self, spec):
        """spec: (B, N, 2) = [m/z, intensity], zero-padded. Returns prefix (B, N, d_dec), mask (B, N)."""
        pmask = spec[..., 0] > 0     # padded peaks have m/z == 0
        with torch.set_grad_enabled(torch.is_grad_enabled() and not self.freeze_encoder):
            h = self.encoder(spec)   # (B, N, d_enc)
        prefix = self.proj(h)
        # move padding to the LEFT so real prefix tokens sit right next to BOS
        order = torch.sort(pmask.int(), dim=1, stable=True).indices
        prefix = prefix.gather(1, order.unsqueeze(-1).expand_as(prefix))
        pmask = pmask.gather(1, order)
        return prefix, pmask

    def forward(self, spec, input_ids, attention_mask, labels=None):
        prefix, pmask = self.encode(spec)
        tok = self.decoder.get_input_embeddings()(input_ids)
        x = torch.cat([prefix.to(tok.dtype), tok], dim=1)
        mask = torch.cat([pmask.long(), attention_mask.long()], dim=1)
        pos = (mask.cumsum(1) - 1).clamp(min=0)       # positions ignore padding
        if labels is not None:                        # no loss on the prefix
            ignore = torch.full(pmask.shape, -100, dtype=labels.dtype, device=labels.device)
            labels = torch.cat([ignore, labels], dim=1)
        return self.decoder(inputs_embeds=x, attention_mask=mask,
                            position_ids=pos, labels=labels)

    @torch.no_grad()
    def generate(self, spec, bos_id, **gen_kwargs):
        prefix, pmask = self.encode(spec)
        B = spec.shape[0]
        bos = self.decoder.get_input_embeddings()(
            torch.full((B, 1), bos_id, device=spec.device))
        x = torch.cat([prefix.to(bos.dtype), bos], dim=1)
        mask = torch.cat([pmask.long(), torch.ones(B, 1, dtype=torch.long, device=spec.device)], dim=1)
        return self.decoder.generate(inputs_embeds=x, attention_mask=mask, **gen_kwargs)

testing out generation

In [2]:
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
from dreams.api import PreTrainedModel
from dreams.models.dreams.dreams import DreaMS as DreaMSModel
from dreams.definitions import PRETRAINED

device = "cuda" if torch.cuda.is_available() else "cpu"

# --- load pieces ---
encoder = PreTrainedModel.from_ckpt(
    ckpt_path=str(Path(PRETRAINED) / "ssl_model.ckpt"),
    ckpt_cls=DreaMSModel, n_highest_peaks=60,
).model
repo = "chandar-lab/NovoMolGen_157M_SMILES_AtomWise"
decoder = AutoModelForCausalLM.from_pretrained(repo, revision="hf-checkpoint")
tokenizer = AutoTokenizer.from_pretrained(repo, revision="hf-checkpoint")
bos, eos = tokenizer.bos_token_id, tokenizer.eos_token_id

model = SpecToSmiles(encoder, decoder).to(device).train()
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable params: {n_train/1e6:.1f}M")

# --- fake spectra: (B, 61, 2) = precursor + 60 peaks, with right-padding ---
B, N = 4, 61
spec = torch.stack([100 + 900 * torch.rand(B, N), torch.rand(B, N)], dim=-1)
for i, n_real in enumerate([61, 40, 20, 8]):   # different peak counts
    spec[i, n_real:] = 0
spec = spec.to(device)

# --- real SMILES through the real tokenizer ---
smiles = ["CCO", "c1ccccc1C(=O)O", "CC(=O)Nc1ccc(O)cc1", "CN1C=NC2=C1C(=O)N(C(=O)N2C)C"]
ids = [[bos] + tokenizer(s, add_special_tokens=False)["input_ids"] + [eos] for s in smiles]
print("round trip ok:", [tokenizer.decode(x[1:-1]).replace(" ", "") == s for x, s in zip(ids, smiles)])

T = max(len(x) for x in ids)
input_ids = torch.tensor([x + [eos] * (T - len(x)) for x in ids])
attention_mask = torch.tensor([[1] * len(x) + [0] * (T - len(x)) for x in ids])
labels = input_ids.clone()
labels[attention_mask == 0] = -100   # mask by attention mask, not by id (pad == eos here)
labels[:, 0] = -100                  # don't predict BOS from the prefix
input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)

# --- 1) forward + loss ---
out = model(spec, input_ids, attention_mask, labels)
print("loss:", out.loss.item(), "| logits:", tuple(out.logits.shape))  # expect (B, 61+T, vocab)

# --- 2) gradients flow only where they should (encoder and decoder frozen) ---
out.loss.backward()
print("proj grad:", all(p.grad is not None for p in model.proj.parameters()))
print("decoder grad:", any(p.grad is not None for p in model.decoder.parameters()))  # expect False
print("encoder grad:", any(p.grad is not None for p in model.encoder.parameters()))  # expect False

# --- 3) generation through inputs_embeds (the code path that broke with GP-MoLFormer) ---
model.eval()
gen = model.generate(spec, bos_id=bos, do_sample=True, max_new_tokens=40,
                     pad_token_id=eos)
print([s.replace(" ", "") for s in tokenizer.batch_decode(gen, skip_special_tokens=True)])

trainable params: 1.1M
round trip ok: [True, True, True, True]
loss: 1.6527589559555054 | logits: (4, 91, 84)
proj grad: True
decoder grad: False
encoder grad: False
['CCOc1ccccc1Cc1ccccnc1', 'C(N(C)C)=C[C@H]()[C@@H]1CC1', 'Cc1ccc(C(=O)N2C[C@](C)(CNC(=O)c3cc(CO)n(C)n', 'Cc1cc(C(=O)N[C@H]2CCN(C(=O)c3[C@H]3CC(c4ccccc4)=']


In [5]:
# 1) scale comparison
E = model.decoder.get_input_embeddings().weight
with torch.no_grad():
    model.proj[-1].weight.fill_(E.std().item())
    model.proj[-1].bias.zero_()
with torch.no_grad():
    prefix, m = model.encode(spec)
print("token emb  std:", E.std().item(), "| mean norm:", E.norm(dim=-1).mean().item())
print("prefix     std:", prefix[m].std().item(), "| mean norm:", prefix[m].norm(dim=-1).mean().item())

# 2) validity with vs without the prefix (RDKit)
from rdkit import Chem, RDLogger
RDLogger.DisableLog("rdApp.*")
valid = lambda ss: sum(s != "" and Chem.MolFromSmiles(s) is not None for s in ss) / len(ss)
dec = lambda g: [s.replace(" ", "") for s in tokenizer.batch_decode(g, skip_special_tokens=True)]

n = 256
g1 = model.generate(spec[:1].repeat(n, 1, 1), bos_id=bos, do_sample=True,
                    max_new_tokens=100, pad_token_id=eos)
ids0 = torch.full((n, 1), bos, device=device)
g0 = model.decoder.generate(input_ids=ids0, attention_mask=torch.ones_like(ids0),
                            do_sample=True, max_new_tokens=100, pad_token_id=eos)
print("validity with prefix:", valid(dec(g1)), "| plain BOS baseline:", valid(dec(g0)))

token emb  std: 0.039221346378326416 | mean norm: 0.8918359279632568
prefix     std: 0.0392172709107399 | mean norm: 0.9921212792396545
validity with prefix: 0.33984375 | plain BOS baseline: 1.0
